In [ ]:
import pandas as pd

df = pd.read_csv("~/predictive-maintenance-xgboost-shap/data/raw/ai4i2020.csv")

In [ ]:
df.info()

In [ ]:
df_clean = df.drop(columns=["UDI", "Product ID", "TWF", "HDF", "PWF", "OSF", "RNF"])
df_clean.to_csv('../data/processed/machine-failure.csv', index=False)

In [ ]:
df_clean['Machine failure'].value_counts()

In [ ]:
df_clean.groupby('Machine failure').mean(numeric_only=True)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.hist(
    df_clean[df_clean['Machine failure'] == 0]['Tool wear [min]'],
    bins=30,
    alpha=0.6,
    label='Sem falha'
)

plt.hist(
    df_clean[df_clean['Machine failure'] == 1]['Tool wear [min]'],
    bins=30,
    alpha=0.6,
    label='Com falha'
)

plt.xlabel('Tool wear [min]')
plt.ylabel('Quantidade de máquinas')
plt.title('Distribuição do desgaste da ferramenta')
plt.legend()
plt.grid(alpha=0.2)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.hist(
    df_clean[df_clean['Machine failure'] == 0]['Tool wear [min]'],
    bins=30,
    density=True,
    alpha=0.6,
    label='Sem falha'
)

plt.hist(
    df_clean[df_clean['Machine failure'] == 1]['Tool wear [min]'],
    bins=30,
    density=True,
    alpha=0.6,
    label='Com falha'
)

plt.xlabel('Tool wear [min]')
plt.ylabel('Densidade')
plt.title('Distribuição do desgaste por ocorrência de falha')
plt.legend()
plt.grid(alpha=0.2)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.hist(
    df_clean[df_clean['Machine failure'] == 0]['Torque [Nm]'],
    bins=30,
    density=True,
    alpha=0.6,
    label='Sem falha'
)

plt.hist(
    df_clean[df_clean['Machine failure'] == 1]['Torque [Nm]'],
    bins=30,
    density=True,
    alpha=0.6,
    label='Com falha'
)

plt.xlabel('Torque [Nm]')
plt.ylabel('Densidade')
plt.title('Distribuição do torque por ocorrência de falha')
plt.legend()
plt.grid(alpha=0.2)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.hist(
    df_clean[df_clean['Machine failure'] == 0]['Rotational speed [rpm]'],
    bins=30,
    density=True,
    alpha=0.6,
    label='Sem falha'
)

plt.hist(
    df_clean[df_clean['Machine failure'] == 1]['Rotational speed [rpm]'],
    bins=30,
    density=True,
    alpha=0.6,
    label='Com falha'
)

plt.xlabel('Rotational speed [rpm]')
plt.ylabel('Densidade')
plt.title('Distribuição do torque por ocorrência de falha')
plt.legend()
plt.grid(alpha=0.2)

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_rpm = df_clean.copy()

df_rpm['RPM faixa'] = pd.cut(
    df_rpm['Rotational speed [rpm]'],
    bins=10
)

failure_rate = df_rpm.groupby(
    'RPM faixa',
    observed=True
)['Machine failure'].mean() * 100

plt.figure(figsize=(10, 6))

failure_rate.plot(kind='bar')

plt.xlabel('Faixa de RPM')
plt.ylabel('Taxa de falha (%)')
plt.title('Taxa de falha por faixa de velocidade de rotação')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.2)

plt.show()

In [ ]:
rpm_analysis = df_rpm.groupby(
    'RPM faixa',
    observed=True
)['Machine failure'].agg(['count', 'sum', 'mean'])

rpm_analysis['failure_rate_%'] = rpm_analysis['mean'] * 100

rpm_analysis

In [ ]:
X = df_clean[[
    'Type',
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]'
]]

y = df_clean['Machine failure']

In [ ]:
X.shape
#10000 maquinas e 6 caracteristicas

In [ ]:
X.shape
y.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
categorical_features = ['Type']

numeric_features = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]'
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)

In [ ]:
X_train_encoded = preprocessor.fit_transform(X_train)

In [ ]:
X_test_encoded = preprocessor.transform(X_test)

In [ ]:
print(X_train_encoded.shape)
print(X_test_encoded.shape)

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier()

In [ ]:
model.fit(X_train_encoded, y_train)

In [ ]:
y_pred = model.predict(X_test_encoded)

In [ ]:
y_pred[:20]

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

cm

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

In [ ]:
precision = precision_score(y_test, y_pred)

print(f"Precision: {precision:.4f}")

In [ ]:
recall = recall_score(y_test, y_pred)

print(f"Recall: {recall:.4f}")

In [ ]:
f1 = f1_score(y_test, y_pred)

print(f"F1-score: {f1:.4f}")

In [ ]:
import shap

In [ ]:
explainer = shap.TreeExplainer(model)

In [ ]:
shap_values = explainer(X_test_encoded)

In [ ]:
explainer = shap.TreeExplainer(model)

In [ ]:
shap_values = explainer(X_test_encoded)

In [ ]:
shap.plots.beeswarm(shap_values, max_display=8)

In [ ]:
feature_names = preprocessor.get_feature_names_out()

print(feature_names)

In [ ]:
X_test_shap = pd.DataFrame(
    X_test_encoded,
    columns=feature_names
)

In [ ]:
feature_names_safe = [
    'Type_H',
    'Type_L',
    'Type_M',
    'Air_temperature_K',
    'Process_temperature_K',
    'Rotational_speed_rpm',
    'Torque_Nm',
    'Tool_wear_min'
]

X_test_shap = pd.DataFrame(
    X_test_encoded,
    columns=feature_names_safe
)
shap_values = explainer(X_test_shap)

In [ ]:
shap.plots.beeswarm(shap_values, max_display=8)